# FastText vs. Word2Vec: `ar` and `zh` on the Ubuntu machine

Linux-specific run configuration for the remaining two languages in `cl.EXTENDED_LANGUAGES`
that haven't started training yet -- `ar` (Arabic) and `zh` (Chinese). `he` (Hebrew) is
running unattended on the Mac tonight, so this notebook is scoped to just `ar`/`zh` to run
in parallel on this machine instead.

Real per-language totals from the 9 languages already finished (`af, eu, hi, kk, ko, mk,
ta, th, vi`) put a full 20-config sweep at roughly 20 hours for `ar` (extrapolated from
`he`'s real corpus-size-scaled total, similar Semitic morphology/script density) and
somewhere in the 2-10 hour range for `zh` -- genuinely uncertain, since `zh` needs
`jieba`-based word segmentation instead of whitespace splitting, the same situation `th`
was in, and `th` trained dramatically faster than its raw corpus size (1.14GB) would
suggest for a whitespace-delimited language that size. Running both here, in parallel with
each other and with `he` on the Mac, is the point of splitting this out.

**Everything below is identical in logic to `compare_algorithms.ipynb`** -- same
`compare_lib.py`, same resumability, same output format -- just: (a) scoped to `ar`/`zh`
only, (b) `workers` set explicitly for this machine's 32 cores rather than relying on the
auto-detected default, and (c) a note below about copying already-built corpus/count files
over from the Mac to skip redundant download+preprocessing time.


## Before running: copy over what's already built (optional, saves hours)

`ar` and `zh`'s corpora are already downloaded, cleaned, pruned, and concatenated on the
Mac (`corpora/corpus-ar.txt` at ~3.7GB, `corpora/corpus-zh.txt` at ~1.9GB), and their
frequency counts may already exist too. If this machine has its own fresh clone of the
repo, `ensure_corpus()`/`ensure_counts()` below will happily download and rebuild
everything from scratch -- correct, but it adds the full download+preprocessing time on
top of the training estimates above for no reason, since the outputs would be identical.

To skip straight to training, copy these paths from the Mac's repo into the same relative
locations in this machine's clone before running the cell below (`rsync -avP` over
SSH/network share, or however this machine's storage is reachable from the Mac):

```
corpora/corpus-ar.txt
corpora/corpus-zh.txt
eval_inputs/counts/ar.subs.2018.tsv.zip
eval_inputs/counts/ar.wiki.2018.tsv.zip
eval_inputs/counts/zh.subs.2018.tsv.zip
eval_inputs/counts/zh.wiki.2018.tsv.zip
```

If `zh`'s counts aren't already built on the Mac, `ensure_counts()` will build them here
instead -- `jieba` (this project's Chinese segmenter, see `requirements.txt`) needs to be
installed either way, so that step works fine locally if the copy above is skipped.


In [ ]:
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
REPO_ROOT = os.path.abspath(os.path.join(HERE, "..", ".."))
sys.path.insert(0, HERE)
import compare_lib as cl

cl.basedir = REPO_ROOT

version = "2018"
configs = [(50, 1), (100, 2), (200, 3), (300, 4), (500, 6)]

# This machine has 32 physical cores available -- set explicitly rather than
# relying on model_training.py's auto default (cpu_count() - 1), so training
# uses the full machine regardless of what gets detected in a container/VM.
workers = 32

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"workers   = {workers}")


## Run `ar` and `zh`

Same `run_comparison_batch()` as the main notebook -- builds each language's corpus/counts
if they're not already there (see the copy note above), trains, scores, and writes
`results/{language}_2018_{timing,rsa,predictive}.csv` plus `results/batch_summary.csv`
incrementally as it goes (safe to check progress mid-run), then regenerates `REPORT.md`
at the end covering every language that has data on disk at that point -- including
whatever's finished on the Mac by then, if this machine can see the same results
directory (shared drive/sync) rather than a separate local clone.

This is the long unattended step -- expect somewhere in the range of a day for `ar` alone
per the estimate above, with `zh` genuinely uncertain but likely faster. Consider running
this via `jupyter nbconvert --to notebook --execute --inplace` under `nohup`/`screen`/`tmux`
rather than an interactive kernel, so it survives an SSH disconnect.


In [ ]:
languages = ["ar", "zh"]
summary = cl.run_comparison_batch(languages, version=version, configs=configs, workers=workers)


### View the generated report

In [ ]:
from IPython.display import Markdown, display

report_path = os.path.join(REPO_ROOT, "experiments", "fasttext_vs_word2vec", "REPORT.md")
with open(report_path) as f:
    display(Markdown(f.read()))
